<a href="https://colab.research.google.com/github/dagracarui25-hash/regbridge/blob/main/notebooks/RegBridge%20%E2%80%94%20Step%202%20%3A%20Full%20Server%20Build.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏦 RegBridge — Step 2: Full Server Build

## Available Endpoints

| Endpoint | Description |
|----------|-------------|
| `POST /question` | Query FINMA circulars only |
| `POST /question-croisee` | Cross-query: FINMA + internal documents |
| `POST /upload-document` | Upload an internal PDF from Lovable |
| `GET /documents` | List all indexed internal documents |
| `DELETE /documents/{name}` | Delete an internal document |

> ⚠️ **Prerequisite**: Step 1 must be completed first
> ([RegBridge — Step 1: FINMA PDF Ingestion](link))
> → Run only once — FINMA MVP documents are already indexed in Qdrant ✅

### 🏦 RegBridge — Step 2 : Build Serveur Complet
### Version enrichie avec upload de documents internes depuis Lovable
### Nouveaux endpoints :
### - `POST /question` → question sur FINMA uniquement
### - `POST /question-croisee` → question sur FINMA + docs internes
### - `POST /upload-document` → upload d'un PDF interne depuis Lovable
### - `GET /documents` → liste des documents internes indexés
### - `DELETE /documents/{nom}` → supprime un document interne
### ⚠️ Prérequis : Étapes 1 terminées (RegBridge — Step 1 : Ingestion des PDFs FINMA) → Une seule fois (Mvp document Finma déjà uploader)


In [ ]:

# ─────────────────────────────────────────────────────────────
# 📦 CELLULE 1 — Installation
# ─────────────────────────────────────────────────────────────
!pip install qdrant-client langchain langchain-community langchain-huggingface langchain-qdrant langchain-text-splitters langchain-groq sentence-transformers groq fastapi uvicorn pyngrok nest-asyncio python-multipart pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.18
    Uninstalling langchain-core-1.2.18:
      Successfully uninstalled langchain-core-1.2.18
ERROR: pip's depen

In [ ]:
# ─────────────────────────────────────────────────────────────
# 🔑 CELLULE 2 — Clés API via Colab Secrets
# ─────────────────────────────────────────────────────────────
from google.colab import userdata

QDRANT_URL     = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")
HF_TOKEN       = userdata.get("HF_TOKEN")
GROQ_API_KEY   = userdata.get("GROQ_API_KEY")
NGROK_TOKEN    = userdata.get("NGROK_TOKEN")

# Domaine fixe permanent ← LIGNE CRITIQUE
NGROK_DOMAIN = "granolithic-belletristic-bulah.ngrok-free.dev"

print(f"✅ QDRANT_URL     : {QDRANT_URL[:30]}...")
print(f"✅ QDRANT_API_KEY : {'*' * 10}")
print(f"✅ GROQ_API_KEY   : {'*' * 10}")
print(f"✅ NGROK_TOKEN    : {'*' * 10}")
print(f"✅ NGROK_DOMAIN   : {NGROK_DOMAIN}")

✅ QDRANT_URL     : https://c1d1b4d3-dc62-4219-90f...
✅ QDRANT_API_KEY : **********
✅ GROQ_API_KEY   : **********
✅ NGROK_TOKEN    : **********
✅ NGROK_DOMAIN   : granolithic-belletristic-bulah.ngrok-free.dev


In [ ]:
# # ─────────────────────────────────────────────────────────────
# 🔑 CELL 3 — RAG Pipeline Initialization (FINMA & Internal Documents)
#    → Embeddings · Qdrant Vector Stores · LLaMA via Groq · Prompts
# ─────────────────────────────────────────────────────────────
import os
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
# Removed langchain_huggingface and replaced with direct sentence_transformers
from sentence_transformers import SentenceTransformer
from langchain_qdrant import QdrantVectorStore
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from huggingface_hub import login # Import login for explicit token handling
from langchain_core.embeddings import Embeddings # Import Embeddings base class
from typing import List

# Define COLLECTION_FINMA and COLLECTION_COMPANY (assuming these are constants)
COLLECTION_FINMA = "finma_compliance"
COLLECTION_COMPANY = "company_documents"

# Fix GROQ_API_KEY (it was malformed in kernel state)
GROQ_API_KEY = "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" # Corrected GROQ API Key

# Store the original HF_TOKEN from Colab Secrets
colab_hf_token = HF_TOKEN

# Temporarily unset the HUGGINGFACEHUB_API_TOKEN environment variable
# and explicitly disable token usage for Hugging Face Hub to prevent 401 errors
# (Reverting complex env var management and using direct SentenceTransformer instead)
# original_huggingfacehub_api_token = os.environ.get("HUGGINGFACEHUB_API_TOKEN")
# original_hf_hub_enable_token = os.environ.get('HF_HUB_ENABLE_HF_TOKEN')

# if "HUGGINGFACEHUB_API_TOKEN" in os.environ:
#     del os.environ["HUGGINGFACEHUB_API_TOKEN"]
# os.environ['HF_HUB_ENABLE_HF_TOKEN'] = 'False' # Explicitly disable token use for Hugging Face Hub

print("⏳ Connexion à Hugging Face Hub...")
# login(token=colab_hf_token) # Commented out to prevent potential authentication conflicts for public model download
print("✅ Connecté à Hugging Face Hub. (Note: login call is temporarily disabled for embedding model loading)")

# Embeddings
  # Ce modèle transforme le texte en vecteurs numériques pour la recherche dans Qdrant.
  # 👍 Multilingue → idéal pour la FINMA (FR/DE/IT/EN).
print("⏳ Chargement embeddings...")
# Using direct SentenceTransformer for robustness
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Custom Embeddings class to wrap SentenceTransformer for LangChain compatibility
class CustomSentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model):
        self.model = model

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        # SentenceTransformer.encode returns numpy arrays, convert to list
        return self.model.encode(texts).tolist()

    def embed_query(self, text: str) -> List[float]:
        # SentenceTransformer.encode returns numpy arrays, convert to list
        return self.model.encode([text])[0].tolist()

# Create an instance of our custom wrapper
langchain_embedding_wrapper = CustomSentenceTransformerEmbeddings(embedding_model)

# Restore the environment variables (no longer needed after direct SentenceTransformer)
# if original_huggingfacehub_api_token is not None:
#     os.environ["HUGGINGFACEHUB_API_TOKEN"] = original_huggingfacehub_api_token
# else:
#     if colab_hf_token:
#         os.environ["HUGGINGFACEHUB_API_TOKEN"] = colab_hf_token

# if original_hf_hub_enable_token is not None:
#     os.environ['HF_HUB_ENABLE_HF_TOKEN'] = original_hf_hub_enable_token
# else:
#     if 'HF_HUB_ENABLE_HF_TOKEN' in os.environ:
#         del os.environ['HF_HUB_ENABLE_HF_TOKEN']

# Client Qdrant (Tu te connectes à ta base vectorielle.)
qdrant_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

# Collection FINMA (Tu accèdes à la collection déjà peuplée contenant les circulaires FINMA embedées.)
print("⏳ Connexion collection FINMA...")
vector_store_finma = QdrantVectorStore.from_existing_collection(
    embedding=langchain_embedding_wrapper,
    url=QDRANT_URL, api_key=QDRANT_API_KEY,
    collection_name=COLLECTION_FINMA,
)

# Collection Company (crée si elle n'existe pas)
print("⏳ Connexion collection documents internes...")
existing = [c.name for c in qdrant_client.get_collections().collections]
if COLLECTION_COMPANY not in existing:
    qdrant_client.create_collection(
        collection_name=COLLECTION_COMPANY,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE)
    )
    print(f"  ✅ Collection '{COLLECTION_COMPANY}' créée (vide)")

vector_store_company = QdrantVectorStore.from_existing_collection(
    embedding=langchain_embedding_wrapper,
    url=QDRANT_URL, api_key=QDRANT_API_KEY,
    collection_name=COLLECTION_COMPANY,
)

# LLM Groq => Initialisation du LLM Groq (LLaMA 3.1 instant)
  # (LLM rapide et peu coûteux → parfait pour une API RAG.)
print("⏳ Connexion Groq/LLaMA 3...")
llm = ChatGroq(api_key=GROQ_API_KEY, model_name="llama-3.1-8b-instant", temperature=0)

# ── Prompts multilingues FR / EN / DE / IT ──
  # Répond uniquement avec les documents FINMA.
    # Refuse tout hallucination.
    # Cite systématiquement les sources.

PROMPTS = {
    "FR": PromptTemplate(
        input_variables=["context", "question"],
        template="""
Tu es un expert en conformité bancaire suisse (compliance officer FINMA).
Réponds UNIQUEMENT avec les informations des documents fournis.
Si l'information n'est pas dans les documents, dis :
"Cette information ne figure pas dans les documents disponibles."
Cite toujours ta source (nom du document + page).
Réponds OBLIGATOIREMENT en français. Sois précis et concis.

DOCUMENTS FINMA:
{context}

QUESTION : {question}
RÉPONSE :"""
    ),
    "EN": PromptTemplate(
        input_variables=["context", "question"],
        template="""
You are a Swiss banking compliance expert (FINMA compliance officer).
Answer ONLY using information from the provided documents.
If the information is not in the documents, say:
"This information is not available in the provided documents."
Always cite your source (document name + page number).
Answer ONLY in English. Be precise and concise.

DOCUMENTS FINMA:
{context}

QUESTION: {question}
ANSWER:"""
    ),
    "DE": PromptTemplate(
        input_variables=["context", "question"],
        template="""
Sie sind ein Schweizer Banken-Compliance-Experte (FINMA Compliance Officer).
Antworten Sie NUR mit Informationen aus den bereitgestellten Dokumenten.
Wenn die Information nicht vorhanden ist, sagen Sie:
"Diese Information ist in den verfügbaren Dokumenten nicht vorhanden."
Zitieren Sie immer Ihre Quelle (Dokumentname + Seitenzahl).
Antworten Sie NUR auf Deutsch. Seien Sie präzise und prägnant.

DOKUMENTE FINMA:
{context}

FRAGE: {question}
ANTWORT:"""
    ),
    "IT": PromptTemplate(
        input_variables=["context", "question"],
        template="""
Sei un esperto di conformità bancaria svizzera (compliance officer FINMA).
Rispondi SOLO con le informazioni dei documenti forniti.
Se l'informazione non è nei documenti, di':
"Questa informazione non è disponibile nei documenti forniti."
Cita sempre la tua fonte (nome documento + pagina).
Rispondi OBBLIGATORIAMENTE in italiano. Sii preciso e conciso.

DOCUMENTI FINMA:
{context}

DOMANDA: {question}
RISPOSTA:"""
    )
}

# Prompts recherche croisés multilingues
 # Compare FINMA vs documents internes.
    # Identifie les écarts réglementaires.
    # Référence systématique les sources.

PROMPTS_CROISE = {
    "FR": PromptTemplate(
        input_variables=["context", "question"],
        template="""
Tu es un expert en conformité bancaire suisse.
Compare les circulaires FINMA avec les documents internes.
Identifie les écarts entre la réglementation et les procédures internes.
Cite toujours ta source. Réponds en français.

DOCUMENTS (FINMA + Internes) :
{context}

QUESTION : {question}
RÉPONSE :"""
    ),
    "EN": PromptTemplate(
        input_variables=["context", "question"],
        template="""
You are a Swiss banking compliance expert.
Compare FINMA circulars with internal company documents.
Identify gaps between regulations and internal procedures.
Always cite your source. Answer in English.

DOCUMENTS (FINMA + Internal):
{context}

QUESTION: {question}
ANSWER:"""
    ),
    "DE": PromptTemplate(
        input_variables=["context", "question"],
        template="""
Sie sind ein Schweizer Banken-Compliance-Experte.
Vergleichen Sie FINMA-Rundschreiben mit internen Dokumenten.
Identifizieren Sie Lücken zwischen Vorschriften und internen Verfahren.
Zitieren Sie immer Ihre Quelle. Antworten Sie auf Deutsch.

DOKUMENTE (FINMA + Intern):
{context}

FRAGE: {question}
ANTWORT:"""
    ),
    "IT": PromptTemplate(
        input_variables=["context", "question"],
        template="""
Sei un esperto di conformità bancaria svizzera.
Confronta le circolari FINMA con i documenti interni aziendali.
Identifica le lacune tra normativa e procedure interne.
Cita sempre la fonte. Rispondi in italiano.

DOCUMENTI (FINMA + Interni):
{context}

DOMANDA: {question}
RISPOSTA:"""
    )
}

print("✅ Prompts multilingues chargés — FR / EN / DE / IT")
print(f"   🏛️  Collection FINMA    : {COLLECTION_FINMA}")
print(f"   🏢  Collection Interne  : {COLLECTION_COMPANY}")


# Agents RAG (Création de l’agent RAG FINMA)➡️ C’est l’agent RAG opérationnel.
    # interroge la collection FINMA
    # récupère les 3 documents les plus pertinents
    # applique ton prompt FINMA
    # génère une réponse réglementaire propre et sourcée
#agent_finma = RetrievalQA.from_chain_type(
    #llm=llm, chain_type="stuff",
    #retriever=vector_store_finma.as_retriever(search_kwargs={"k": 3}),
    #return_source_documents=True,
   # vchain_type_kwargs={"prompt": prompt_finma}
#)

print("\n✅ Tout est prêt !")
print(f"   🏛️  Collection FINMA    : {COLLECTION_FINMA}")
print(f"   🏢  Collection Interne  : {COLLECTION_COMPANY}")

⏳ Connexion à Hugging Face Hub...
✅ Connecté à Hugging Face Hub. (Note: login call is temporarily disabled for embedding model loading)
⏳ Chargement embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⏳ Connexion collection FINMA...
⏳ Connexion collection documents internes...
⏳ Connexion Groq/LLaMA 3...
✅ Prompts multilingues chargés — FR / EN / DE / IT
   🏛️  Collection FINMA    : finma_compliance
   🏢  Collection Interne  : company_documents

✅ Tout est prêt !
   🏛️  Collection FINMA    : finma_compliance
   🏢  Collection Interne  : company_documents


In [ ]:
from huggingface_hub import HfApi
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    print("❌ HF_TOKEN n'est pas défini dans les secrets Colab.")
else:
    try:
        api = HfApi(token=HF_TOKEN)
        user_info = api.whoami()
        print("✅ HF_TOKEN semble valide. Informations de l'utilisateur Hugging Face :")
        print(f"  Nom d'utilisateur: {user_info.get('name', 'N/A')}")
        print(f"  Pseudo: {user_info.get('preferred_username', 'N/A')} (peut être absent pour certains comptes)")
        print(f"  E-mail: {user_info.get('email', 'N/A')}")
        print(f"  Organisations: {[org['name'] for org in user_info.get('orgs', [])]}")
    except Exception as e:
        print(f"❌ HF_TOKEN semble invalide ou a des permissions insuffisantes. Erreur: {e}")
        print("Veuillez vérifier votre HF_TOKEN dans les secrets Colab et assurez-vous qu'il a les permissions nécessaires, notamment pour télécharger des modèles.")

✅ HF_TOKEN semble valide. Informations de l'utilisateur Hugging Face :
  Nom d'utilisateur: dagracarui25
  Pseudo: N/A (peut être absent pour certains comptes)
  E-mail: N/A
  Organisations: []


In [ ]:
# ─────────────────────────────────────
# 🔍 DIAGNOSTIC — Affiche le Contenu des collections Qdrant
# ─────────────────────────────────────
from qdrant_client import QdrantClient
import os

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

# Liste des collections
collections = client.get_collections().collections
print("📦 COLLECTIONS DISPONIBLES :")
for c in collections:
    count = client.count(c.name).count
    print(f"   • {c.name} → {count} chunks")

print()

# Détail collection FINMA
print("🏛️  FICHIERS FINMA :")
finma_points = client.scroll(
    collection_name="finma_compliance",
    with_payload=True, limit=500
)[0]
finma_docs = {}
for p in finma_points:
    src = p.payload.get("metadata", {}).get("source", "?")
    fname = os.path.basename(src)
    finma_docs[fname] = finma_docs.get(fname, 0) + 1
for fname, chunks in finma_docs.items():
    print(f"   • {fname} ({chunks} chunks)")

print()

# Détail collection interne
print("🏢  FICHIERS INTERNES :")
company_points = client.scroll(
    collection_name="company_documents",
    with_payload=True, limit=500
)[0]
company_docs = {}
for p in company_points:
    src = p.payload.get("metadata", {}).get("source", "?")
    company_docs[src] = company_docs.get(src, 0) + 1
for fname, chunks in company_docs.items():
    print(f"   • {fname} ({chunks} chunks)")

📦 COLLECTIONS DISPONIBLES :
   • finma_compliance → 1377 chunks
   • company_documents → 7082 chunks

🏛️  FICHIERS FINMA :
   • fedlex-data-admin-ch-eli-cc-2015-791-20230101-de-pdf-a (1).pdf (29 chunks)
   • finma rs 2023 01 20221207 IT (2).pdf (36 chunks)
   • finma rs 2011 01 01 01 2017 DE (1).pdf (57 chunks)
   • fedlex-data-admin-ch-eli-cc-2015-791-20230101-it-pdf-a (1).pdf (29 chunks)
   • finma rs 2023 01 20221207 DE (2).pdf (46 chunks)
   • finma rs 2023 01 20221207 EN (2).pdf (31 chunks)
   • finma rs 2023 01 20221207 FR.pdf (33 chunks)
   • fedlex-data-admin-ch-eli-cc-2015-791-20230101-fr-pdf-a (1).pdf (30 chunks)
   • finma rs 2011 01 01 01 2017 IT (1).pdf (66 chunks)
   • finma rs 2016 07 20210506 IT (2).pdf (25 chunks)
   • finma rs 2016 07 20210506 DE (2).pdf (21 chunks)
   • finma rs 2016 07 20210506 FR.pdf (22 chunks)
   • finma rs 2011 01 01 01 2017 FR (1).pdf (55 chunks)
   • finma rs 2016 07 20210506 EN (2).pdf (20 chunks)

🏢  FICHIERS INTERNES :
   • FATF Recommendat

In [ ]:
# ─────────────────────────────────────────────────────────────
# 🌐 CELLULE 4 — Création du serveur FastAPI
# ─────────────────────────────────────────────────────────────
import os, tempfile, shutil, json
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document as LCDocument
from typing import Optional

app = FastAPI(
    title="RegBridge API",
    description="Assistant Conformité FINMA + Documents Internes",
    version="2.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Modèles Pydantic ──
class Question(BaseModel):
    question: str
    mode: Optional[str] = "finma"      # "finma" ou "croise"
    langue: Optional[str] = "FR"       # FR / EN / DE / IT

class Reponse(BaseModel):
    reponse: str
    sources: list
    mode: str

# ── Helpers ──
def extraire_sources(docs):
    sources, vues = [], []
    for doc in docs:
        source = doc.metadata.get("source", "Document inconnu")
        page   = doc.metadata.get("page", "?")
        cat    = doc.metadata.get("categorie", "FINMA")
        ref    = f"{source}-{page}"
        if ref not in vues:
            vues.append(ref)
            sources.append({
                "document": source,
                "page": page,
                "categorie": cat,
                "extrait": doc.page_content[:250]
            })
    return sources

def rebuild_agent_croise(langue="FR"):
    """Reconstruit le retriever croisé avec les 2 collections Qdrant."""
    from langchain.retrievers import MergerRetriever
    retriever_finma   = vector_store_finma.as_retriever(search_kwargs={"k": 2})
    retriever_company = vector_store_company.as_retriever(search_kwargs={"k": 2})
    return MergerRetriever(retrievers=[retriever_finma, retriever_company])

# ════════════════════════════════════════════
# ENDPOINT 1 — Santé
# ════════════════════════════════════════════
@app.get("/")
async def health():
    existing      = [c.name for c in qdrant_client.get_collections().collections]
    count_company = qdrant_client.count(COLLECTION_COMPANY).count if COLLECTION_COMPANY in existing else 0
    count_finma   = qdrant_client.count(COLLECTION_FINMA).count   if COLLECTION_FINMA   in existing else 0
    return {
        "status": "✅ RegBridge API v2.0 opérationnelle",
        "collections": {
            "finma_compliance":  f"{count_finma} chunks",
            "company_documents": f"{count_company} chunks"
        }
    }

# ════════════════════════════════════════════
# ENDPOINT 2 — Question (FINMA ou Croisée)
# ════════════════════════════════════════════
@app.post("/question", response_model=Reponse)
async def poser_question(body: Question):
    langue = body.langue.upper() if body.langue else "FR"
    if langue not in PROMPTS:
        langue = "FR"

    if body.mode == "croise":
        prompt = PROMPTS_CROISE[langue]
        from langchain.retrievers import MergerRetriever
        retriever = MergerRetriever(retrievers=[
            vector_store_finma.as_retriever(search_kwargs={"k": 2}),
            vector_store_company.as_retriever(search_kwargs={"k": 2})
        ])
    else:
        prompt    = PROMPTS[langue]
        retriever = vector_store_finma.as_retriever(search_kwargs={"k": 3})

    retrieved_docs = retriever.invoke(body.question)

    context_parts = []
    for doc in retrieved_docs:
        source   = doc.metadata.get("source", "Document inconnu")
        filename = os.path.basename(source)
        page     = doc.metadata.get("page", "?")
        context_parts.append(f"[{filename} — page {page}]\n{doc.page_content}")
    context = "\n\n".join(context_parts)

    formatted_prompt = prompt.format(context=context, question=body.question)
    llm_result       = llm.invoke(formatted_prompt)
    sources          = extraire_sources(retrieved_docs)

    return Reponse(
        reponse=llm_result.content,
        sources=sources,
        mode=body.mode or "finma"
    )

# ════════════════════════════════════════════
# ENDPOINT 3 — Upload document interne
# ════════════════════════════════════════════
@app.post("/upload-document")
async def upload_document(
    file:      UploadFile = File(...),
    categorie: str        = Form(default="Procédure interne")
):
    if not file.filename.lower().endswith(".pdf"):
        raise HTTPException(status_code=400, detail="Seuls les fichiers PDF sont acceptés.")

    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
        shutil.copyfileobj(file.file, tmp)
        tmp_path = tmp.name

    loader = PyPDFLoader(tmp_path)
    docs   = loader.load()

    for doc in docs:
        doc.metadata.update({
            "source":       file.filename,
            "categorie":    categorie,
            "type":         "document_interne",
            "confidential": True
        })

    splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)
    chunks   = splitter.split_documents(docs)

    vector_store_company.add_documents(chunks)
    os.unlink(tmp_path)

    return {
        "status":    "✅ Document ingéré avec succès",
        "filename":  file.filename,
        "categorie": categorie,
        "pages":     len(docs),
        "chunks":    len(chunks),
        "message":   f"{len(chunks)} passages indexés et disponibles pour la recherche"
    }

# ════════════════════════════════════════════
# ENDPOINT 4 — Liste documents FINMA + Internes
# ✅ PATCH : retourne les 2 collections séparément
# ════════════════════════════════════════════
@app.get("/documents")
async def liste_documents():
    response = {"finma": [], "company": [], "documents": [], "total_documents": 0}
    try:
        # ── Collection FINMA ──
        finma_points = qdrant_client.scroll(
            collection_name=COLLECTION_FINMA,
            with_payload=True, limit=500
        )[0]
        finma_vus = {}
        for p in finma_points:
            meta  = p.payload.get("metadata", {})
            src   = meta.get("source", "?")
            fname = os.path.basename(src)
            if fname not in finma_vus:
                finma_vus[fname] = {"filename": fname, "categorie": "FINMA", "chunks": 0}
            finma_vus[fname]["chunks"] += 1
        response["finma"] = list(finma_vus.values())

        # ── Collection interne ──
        company_points = qdrant_client.scroll(
            collection_name=COLLECTION_COMPANY,
            with_payload=True, limit=500
        )[0]
        company_vus = {}
        for p in company_points:
            meta = p.payload.get("metadata", {})
            src  = meta.get("source", "?")
            if src not in company_vus:
                company_vus[src] = {
                    "filename":  src,
                    "categorie": meta.get("categorie", "Non classé"),
                    "chunks":    0
                }
            company_vus[src]["chunks"] += 1
        response["company"] = list(company_vus.values())

        # ── Rétrocompatibilité : champ "documents" = tout ──
        response["documents"]       = response["finma"] + response["company"]
        response["total_documents"] = len(response["finma"]) + len(response["company"])

    except Exception as e:
        response["error"] = str(e)

    return response

# ════════════════════════════════════════════
# ENDPOINT 5 — Supprimer un document interne
# ════════════════════════════════════════════
@app.delete("/documents/{filename}")
async def supprimer_document(filename: str):
    from qdrant_client.models import Filter, FieldCondition, MatchValue
    qdrant_client.delete(
        collection_name=COLLECTION_COMPANY,
        points_selector=Filter(
            must=[FieldCondition(
                key="metadata.source",
                match=MatchValue(value=filename)
            )]
        )
    )
    return {"status": f"✅ Document '{filename}' supprimé"}

print("✅ Serveur FastAPI v2.0 configuré")
print("   GET  /                → Santé + stats collections")
print("   POST /question        → Question FINMA ou croisée")
print("   POST /upload-document → Upload PDF interne")
print("   GET  /documents       → Liste FINMA + docs internes ✅")
print("   DEL  /documents/{f}   → Supprimer un document")

✅ Serveur FastAPI v2.0 configuré
   GET  /                → Santé + stats collections
   POST /question        → Question FINMA ou croisée
   POST /upload-document → Upload PDF interne
   GET  /documents       → Liste FINMA + docs internes ✅
   DEL  /documents/{f}   → Supprimer un document


In [ ]:
# 🔧 PATCH — Supprime l'ancienne route /documents et la remplace
from fastapi.routing import APIRoute

# Supprimer l'ancienne route GET /documents
app.routes[:] = [
    r for r in app.routes
    if not (isinstance(r, APIRoute) and r.path == "/documents" and "GET" in r.methods)
]

# Redéfinir la nouvelle route
@app.get("/documents")
async def liste_documents():
    response = {"finma": [], "company": [], "documents": [], "total_documents": 0}
    try:
        finma_points = qdrant_client.scroll(
            collection_name=COLLECTION_FINMA,
            with_payload=True, limit=500
        )[0]
        finma_vus = {}
        for p in finma_points:
            meta  = p.payload.get("metadata", {})
            src   = meta.get("source", "?")
            fname = os.path.basename(src)
            if fname not in finma_vus:
                finma_vus[fname] = {"filename": fname, "categorie": "FINMA", "chunks": 0}
            finma_vus[fname]["chunks"] += 1
        response["finma"] = list(finma_vus.values())

        company_points = qdrant_client.scroll(
            collection_name=COLLECTION_COMPANY,
            with_payload=True, limit=500
        )[0]
        company_vus = {}
        for p in company_points:
            meta = p.payload.get("metadata", {})
            src  = meta.get("source", "?")
            if src not in company_vus:
                company_vus[src] = {
                    "filename":  src,
                    "categorie": meta.get("categorie", "Non classé"),
                    "chunks":    0
                }
            company_vus[src]["chunks"] += 1
        response["company"] = list(company_vus.values())

        response["documents"]       = response["finma"] + response["company"]
        response["total_documents"] = len(response["finma"]) + len(response["company"])

    except Exception as e:
        response["error"] = str(e)

    return response

print("✅ Route /documents remplacée")
print(f"   Routes actives : {[r.path for r in app.routes if isinstance(r, APIRoute)]}")

✅ Route /documents remplacée
   Routes actives : ['/', '/question', '/upload-document', '/documents/{filename}', '/documents']


In [ ]:
# ─────────────────────────────────────────────────────────────
# 🚀 CELLULE 5 — Démarrage Ngrok (domaine fixe) + Serveur
# ─────────────────────────────────────────────────────────────
import nest_asyncio, uvicorn, threading, time, requests
from pyngrok import ngrok, conf

nest_asyncio.apply()

# ── Domaine fixe permanent ──
NGROK_DOMAIN = "granolithic-belletristic-bulah.ngrok-free.dev"

# ── Ferme les anciens tunnels ──
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
        print(f"🧹 Ancien tunnel fermé : {t.public_url}")
except:
    pass

# ── Configure le token ──
conf.get_default().auth_token = NGROK_TOKEN
time.sleep(1)

# ── Lance le tunnel avec domaine fixe ──
try:
    tunnel = ngrok.connect(addr=8000, domain=NGROK_DOMAIN)
    NGROK_URL = tunnel.public_url
    print("=" * 60)
    print("🚀 REGBRIDGE API v2.0 DÉMARRÉE !")
    print("=" * 60)
    print(f"\n🌐 URL FIXE PERMANENTE :")
    print(f"\n   👉 {NGROK_URL}")
    print(f"\n📡 POST /question          → Questions FINMA & croisées")
    print(f"📤 POST /upload-document   → Upload PDF interne")
    print(f"📋 GET  /documents         → Liste documents indexés")
    print(f"📚 GET  /docs              → Documentation Swagger")
    print("=" * 60)
except Exception as e:
    print(f"❌ Erreur ngrok : {e}")
    print("→ Va sur dashboard.ngrok.com → Tunnels → Delete all")

# ── Lance uvicorn en thread non-bloquant ──
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)

# ── Keepalive Pro+ intégré (ping toutes les 90s) ──
def keepalive_loop():
    print(f"\n🟢 Keepalive démarré → ping toutes les 90s")
    erreurs = 0
    while True:
        try:
            r = requests.get(f"{NGROK_URL}/health", timeout=10)
            if r.status_code == 200:
                erreurs = 0
                print(f"[{time.strftime('%H:%M:%S')}] ✅ Serveur OK")
            else:
                erreurs += 1
                print(f"[{time.strftime('%H:%M:%S')}] ⚠️ Status: {r.status_code}")
        except Exception as e:
            erreurs += 1
            print(f"[{time.strftime('%H:%M:%S')}] ❌ Erreur #{erreurs}: {e}")
            if erreurs >= 3:
                print("🔴 ALERTE : 3 erreurs consécutives !")
        time.sleep(90)

ka_thread = threading.Thread(target=keepalive_loop, daemon=True)
ka_thread.start()

ERROR:pyngrok.process.ngrok:t=2026-03-17T15:37:41+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-03-17T15:37:41+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


🚀 REGBRIDGE API v2.0 DÉMARRÉE !

🌐 URL FIXE PERMANENTE :

   👉 https://granolithic-belletristic-bulah.ngrok-free.dev

📡 POST /question          → Questions FINMA & croisées
📤 POST /upload-document   → Upload PDF interne
📋 GET  /documents         → Liste documents indexés
📚 GET  /docs              → Documentation Swagger

🟢 Keepalive démarré → ping toutes les 90s


In [13]:
# ─────────────────────────────────────
# 🔍 CHECK - l'état du Tunnel Ngrok
# ─────────────────────────────────────
from pyngrok import ngrok

try:
    tunnels = ngrok.get_tunnels()
    if tunnels:
        print("Tunnels ngrok actifs :")
        for tunnel in tunnels:
            print(f"  - {tunnel.public_url} ({tunnel.proto})")
    else:
        print("Aucun tunnel ngrok actif.")
except Exception as e:
    print(f"Erreur lors de la récupération des tunnels ngrok : {e}")

Tunnels ngrok actifs :
  - https://granolithic-belletristic-bulah.ngrok-free.dev (https)
